## Building A Chatbot
In this section we will design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation.


In [2]:
import os 
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [3]:
groq_api_key=os.getenv("GROQ_API_KEY")


In [4]:
from langchain_groq import ChatGroq 

model=ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=groq_api_key
)


d:\Gen_Ai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import SystemMessage

model.invoke([HumanMessage(content="Hi , My name is Shivesh and I am a Chief AI Engineer")])

AIMessage(content="Hello Shivesh, nice to meet you. It's great to connect with a Chief AI Engineer like yourself. What brings you here today? Are you working on any exciting AI projects or looking to discuss the latest advancements in the field? I'm all ears.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 50, 'total_tokens': 104, 'completion_time': 0.178421752, 'completion_tokens_details': None, 'prompt_time': 0.004825205, 'prompt_tokens_details': None, 'queue_time': 0.162069603, 'total_time': 0.183246957}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eab05-a3a8-7693-b26e-636edf3f4bf7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 54, 'total_tokens': 104})

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Shivesh and I am a Chief AI Engineer"),
        AIMessage(content="Hello Shivesh! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="Your name is Shivesh, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 121, 'total_tokens': 137, 'completion_time': 0.033731726, 'completion_tokens_details': None, 'prompt_time': 0.01264303, 'prompt_tokens_details': None, 'queue_time': 0.162845169, 'total_time': 0.046374756}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eab05-a68b-7423-956e-05b7a0a293e7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens': 16, 'total_tokens': 137})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input.

In [7]:
!pip install langchain_community

In [8]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory # chat history is imported from this library 
from langchain_core.runnables.history import RunnableWithMessageHistory

# every message that we are putting up needs to be added in the message history 


store={}

# When different different users are chatting over llm model how we are going to ensure that one session
# is completely different from the other session 


# session id will be used to differentiate one session from another
def  get_session_history(session_id:str)->BaseChatMessageHistory :# will create session id (string) .....return type will be chatmessage history
        if session_id not in store: 
            store[session_id]=ChatMessageHistory() # if session id not in store... means new session... so make new object of chatmessagehistory...
        return store[session_id]
    
    
with_message_history=RunnableWithMessageHistory(model,get_session_history) #for particular model and session_history this will give entire chat history for {model,session} pair 

            

D:\Temp\ipykernel_29036\787193593.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
d:\Gen_Ai\venv\lib\site-packages\IPython\core\interactiveshell.py:3577: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [9]:
config={"configurable":{"session_id":"chat1"}}

In [10]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Shivesh and I am a Chief AI Engineer")],
    config=config # for this session id '{config : session_id : chat 1 }' we are interacting 
)

In [11]:
response.content

"Hello Shivesh, nice to meet you. It's great to connect with a Chief AI Engineer like yourself. What brings you here today? Are you working on any exciting AI projects or looking to discuss the latest advancements in the field? I'm all ears and ready to chat."

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="What is my name ? ")],
    config=config # for this session id '{config : session_id : chat 1 }' we are interacting 
)

In [13]:
response.content

'Your name is Shivesh.'

In [14]:
## change the config --> session_id 
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What is my name ? ")],
    config=config1 # for this session id '{config : session_id : chat 2 }' we are interacting 
)
#this time we  have made a new session_id so it should not be able to remember my name 
response.content



"I don't know your name. I'm a large language model, I don't have the ability to know your personal information, including your name. I can only respond based on the text you provide to me. If you'd like to share your name, I'd be happy to chat with you and use it in our conversation!"

In [15]:
response=with_message_history.invoke(
    [HumanMessage(content="My name is john ? ")],
    config=config1 # for this session id '{config : session_id : chat 1 }' we are interacting 
)
response.content

"Nice to meet you, John. However, I should let you know that I don't actually store or remember names, so this is just a temporary greeting. If you interact with me again in the future, I won't recall that your name is John unless you tell me again. But for the purpose of our current conversation, I'd be happy to address you as John! How can I assist you today, John?"

In [16]:
## change the config --> session_id 
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What is my name ? ")],
    config=config1 # for this session id '{config : session_id : chat 2 }' we are interacting 
)

# now this time it will show our name as john 
response.content

# In this way we can switch between context ....based on different session id ....

"You told me earlier that your name is John. However, as I mentioned before, I don't store or remember names, so I'm only recalling it because it's part of our current conversation. If we start a new conversation, I won't remember that your name is John. But for now, I'll still address you as John!"

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [17]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [18]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Shivesh")]})

AIMessage(content="Hello Shivesh, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 58, 'total_tokens': 86, 'completion_time': 0.058731122, 'completion_tokens_details': None, 'prompt_time': 0.005239567, 'prompt_tokens_details': None, 'queue_time': 0.163545251, 'total_time': 0.063970689}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eab05-d380-7b52-a0c4-f7422c10f509-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 28, 'total_tokens': 86})

In [19]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

d:\Gen_Ai\venv\lib\site-packages\IPython\core\interactiveshell.py:3577: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [20]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Shivesh")],
    config=config
)

response.content

"Hello Shivesh, it's nice to meet you. Is there something I can help you with or would you like to chat?"

In [21]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Shivesh.'

In [22]:
## Adding more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [23]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Shivesh")],"language":"Hindi"})
response.content

'नमस्ते शिवेश, मैं आपकी सहायता के लिए तैयार हूँ। मैं आपके प्रश्नों का उत्तर देने की पूरी कोशिश करूँगा। क्या आप कुछ पूछना चाहते हैं या मेरी मदद से कुछ करना चाहते हैं?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [24]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

d:\Gen_Ai\venv\lib\site-packages\IPython\core\interactiveshell.py:3577: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [25]:
config={"configurable":{"session_id":"chat4"}}


repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am shivesh")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते शिवेश, मैं आपकी कैसे मदद कर सकता हूँ?'

In [26]:
response=with_message_history.invoke(
    {'messages':[HumanMessage(content ="What is my name  ? ")],"language":"Hindi"},
    config=config
)

response.content

'आपका नाम शिवेश है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [27]:
!pip install transformers

In [28]:
!pip install torch sentencepiece

In [29]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45, #max token limit to consider 
    strategy="last", # take last messages 
    token_counter=model, 
    include_system=True, # include system message always dont trim that 
    allow_partial=False,
    start_on="human" # start with human message
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

d:\Gen_Ai\venv\lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
d:\Gen_Ai\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate develop

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
#applying concept of trimming on chaining method

from operator import itemgetter 

from langchain_core.runnables import RunnablePassthrough 

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer )
    | prompt 
    | model 
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

# we dont have context of favourite icecream due  trimming due which its not able to ..tell fav icecream 

"I don't have that information. I'm a new conversation, I don't have any prior knowledge about your preferences, including your favorite ice cream flavor. Would you like to tell me?"

In [ ]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

# It remembers 2+2 since we havent trimmed that 

'You asked what 2 + 2 is. The answer is 4.'

In [33]:
#Lets wrap this  in message history 


with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}


d:\Gen_Ai\venv\lib\site-packages\IPython\core\interactiveshell.py:3577: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [34]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"I don't know your name, you haven't told me yet. Would you like to share it with me?"